# 주석 영어

In [ ]:
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# 1. Load data

DATA_DIR = "/content/"

PRICE_PROCESSED_PATH = "/content/gsm_processed(price_prediction).csv"
RECO_PROCESSED_PATH = "/content/gsm_processed(recommendation).csv"
PRICE_ALL_PATH = "/content/gsm_processed_all(price_prediction).csv"
RECO_ALL_PATH = "/content/gsm_processed_all(recommendation).csv"

csv_candidates = glob.glob(os.path.join(DATA_DIR, "*.csv"))

if len(csv_candidates) == 0:
    raise FileNotFoundError("'/content/' 폴더 안에 csv 파일이 없습니다.")

DATA_PATH = csv_candidates[0]
print("[INFO] Loaded file:", DATA_PATH)

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("[INFO] Raw shape:", df_raw.shape)

try:
    display(df_raw.head())
except:
    print(df_raw.head())



# 2. Inspect dataset

print("\n[INFO] Columns")
print(df_raw.columns.tolist())

print("\n[INFO] Data types")
try:
    display(df_raw.dtypes)
except:
    print(df_raw.dtypes)

print("\n[INFO] Missing values - top 20")
missing_before = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_ratio": df_raw.isna().mean()
}).sort_values("missing_ratio", ascending=False)

try:
    display(missing_before.head(20))
except:
    print(missing_before.head(20))

print("\n[INFO] duplicated rows:", df_raw.duplicated().sum())



# 3. Common text cleaning functions
# Convert placeholder values to np.nan
# Keep 'No' because it indicates a missing feature, not a missing value

def normalize_missing(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        s = x.strip()
        if s in ["", "-", "—", "–", "N/A", "n/a", "NA", "na", "None", "none", "null", "Null"]:
            return np.nan
        return s

    return x

# Fix corrupted special characters in text
def clean_mojibake(text):
    if pd.isna(text):
        return np.nan

    s = str(text)

    replacements = {
        "<e2><82><ac>": "€",
        "<c2><a3>": "£",
        "<e2><82><b9>": "₹",
        "<e2><80><89>": " ",
        "<c2><a0>": " ",
        "<c2><b5>": "μ",
        "<cb><9a>": "°",
    }

    for k, v in replacements.items():
        s = s.replace(k, v)

    s = re.sub(r"\s+", " ", s).strip()
    return s


df = df_raw.copy()
df = df.applymap(normalize_missing)

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].map(clean_mojibake)

df = df.drop_duplicates().reset_index(drop=True)

print("[INFO] Shape after duplicate removal:", df.shape)

# Return the column if it exists, otherwise return an all-NaN Series
# (Prevent errors when column names differ slightly)
def get_col(col_name):
    if col_name in df.columns:
        return df[col_name]
    return pd.Series(np.nan, index=df.index)



# 4. Parsing functions
def parse_year(text):
    """
    Extract release year from launch_announced
    example: '2019, October 15' -> 2019
    """
    if pd.isna(text):
        return np.nan

    years = re.findall(r"(?:19|20)\d{2}", str(text))
    if len(years) == 0:
        return np.nan

    return int(years[0])


month_map = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,
}


# Extract release month from launch_announced
# Convert quarter values to the middle month of the quarter
def parse_month(text):
    if pd.isna(text):
        return np.nan

    s = str(text).lower()

    for month_name, month_num in month_map.items():
        if month_name in s:
            return month_num

    q = re.search(r"\bq([1-4])\b", s)
    if q:
        return (int(q.group(1)) - 1) * 3 + 2

    return np.nan

# Extract weight in grams from body_weight
def parse_weight_g(text):
    if pd.isna(text):
        return np.nan

    s = str(text).lower().replace(",", "")
    m = re.search(r"(\d+(?:\.\d+)?)\s*g\b", s)

    if m:
        return float(m.group(1))

    return np.nan

# Extract height, width, and thickness from body_dimensions
def parse_dimensions_mm(text):
    if pd.isna(text):
        return (np.nan, np.nan, np.nan)

    s = str(text).lower()
    s = s.split("(")[0]

    m = re.search(
        r"(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*mm",
        s
    )

    if m:
        return tuple(float(m.group(i)) for i in range(1, 4))

    return (np.nan, np.nan, np.nan)

# Extract display size in inches
def parse_display_size_in(text):
    if pd.isna(text):
        return np.nan

    s = str(text).lower()
    m = re.search(r"(\d+(?:\.\d+)?)\s*inches", s)

    if m:
        return float(m.group(1))

    return np.nan

# Extract screen-to-body ratio
def parse_screen_body_ratio(text):
    if pd.isna(text):
        return np.nan

    m = re.search(r"~\s*(\d+(?:\.\d+)?)\s*%", str(text))

    if m:
        return float(m.group(1))

    return np.nan

# Extract width, height, and PPI from display_resolution
def parse_resolution(text):
    if pd.isna(text):
        return (np.nan, np.nan, np.nan)

    s = str(text).lower().replace(",", "")

    m = re.search(r"(\d{2,5})\s*x\s*(\d{2,5})\s*pixels", s)

    width = np.nan
    height = np.nan

    if m:
        width = float(m.group(1))
        height = float(m.group(2))

    p = re.search(r"~\s*(\d+(?:\.\d+)?)\s*ppi", s)
    ppi = float(p.group(1)) if p else np.nan

    return (width, height, ppi)

# Extract battery capacity in mAh
def parse_battery_mah(text):
    if pd.isna(text):
        return np.nan

    s = str(text).lower().replace(",", "")
    m = re.search(r"(\d{3,5})\s*mah", s)

    if m:
        return float(m.group(1))

    return np.nan

# Extract battery type
def parse_battery_type(text):
    if pd.isna(text):
        return "Unknown"

    s = str(text).lower()

    if "li-po" in s or "lipo" in s or "li po" in s:
        return "Li-Po"
    elif "li-ion" in s or "li ion" in s:
        return "Li-Ion"
    else:
        return "Unknown"

# Determine whether the battery is removable 
# removable -> 1
# non-removable -> 0
def parse_removable_battery(text):
    if pd.isna(text):
        return np.nan

    s = str(text).lower()

    if "non-removable" in s or "non removable" in s:
        return 0
    elif "removable" in s:
        return 1
    else:
        return np.nan


# Extract charging wattage from battery_charging
def parse_charging_w(text):
    if pd.isna(text):
        return np.nan

    vals = re.findall(r"(\d+(?:\.\d+)?)\s*w\b", str(text).lower())

    if len(vals) == 0:
        return np.nan

    vals = [float(v) for v in vals]
    return max(vals)

# Convert storage and RAM units to GB
def to_gb(num, unit):
    num = float(str(num).replace(",", ""))
    unit = str(unit).upper()

    if unit == "TB":
        return num * 1024
    elif unit == "GB":
        return num
    elif unit == "MB":
        return num / 1024
    elif unit == "KB":
        return num / (1024 ** 2)
    else:
        return np.nan

# Extract maximum storage and RAM from memory_internal
def parse_memory(text):
    if pd.isna(text):
        return (np.nan, np.nan)

    s = str(text).upper().replace(",", " ")
    s = re.sub(r"\s+", " ", s)

    ram_matches = re.findall(r"(\d+(?:\.\d+)?)\s*(TB|GB|MB|KB)\s*RAM", s)
    ram_values = [to_gb(num, unit) for num, unit in ram_matches]

    storage_text = re.sub(r"\d+(?:\.\d+)?\s*(?:TB|GB|MB|KB)\s*RAM", " ", s)
    storage_matches = re.findall(r"(\d+(?:\.\d+)?)\s*(TB|GB|MB|KB)", storage_text)
    storage_values = [to_gb(num, unit) for num, unit in storage_matches]

    max_storage = max(storage_values) if len(storage_values) > 0 else np.nan
    max_ram = max(ram_values) if len(ram_values) > 0 else np.nan

    return (max_storage, max_ram)

# Extract the largest camera MP value
def max_mp_from_texts(*texts):
    vals = []

    for text in texts:
        if pd.isna(text):
            continue

        s = str(text).lower()
        found = re.findall(r"(\d+(?:\.\d+)?)\s*mp\b", s)
        vals.extend([float(x) for x in found])

    if len(vals) == 0:
        return np.nan

    return max(vals)

# Return 0 for "No", otherwise return 1
def yes_no_feature(text):
    if pd.isna(text):
        return 0

    s = str(text).strip().lower()

    if s == "" or s == "no" or s.startswith("no"):
        return 0

    return 1

# Check whether specific keywords exist
def contains_feature(text, patterns):
    if pd.isna(text):
        return 0

    s = str(text).lower()

    for p in patterns:
        if p in s:
            return 1

    return 0

# Count the number of sensors
def count_sensors(text):
    if pd.isna(text):
        return 0

    s = str(text).strip()

    if s.lower() in ["no", "v1"]:
        return 0

    parts = re.split(r",|;", s)
    parts = [p.strip() for p in parts if p.strip()]

    return len(parts)

# Categorize OS family
def os_family(text):
    if pd.isna(text):
        return "Unknown"

    s = str(text).lower()

    if "android" in s:
        return "Android"
    elif "ios" in s or "iphone os" in s:
        return "iOS"
    elif "windows" in s:
        return "Windows"
    elif "blackberry" in s:
        return "BlackBerry"
    elif "symbian" in s:
        return "Symbian"
    elif "linux" in s:
        return "Linux"
    else:
        return "Other"

# Extract OS version number
def parse_os_version(text):

    if pd.isna(text):
        return np.nan

    s = str(text)

    m = re.search(r"(\d+(?:\.\d+)?)", s)

    if m:
        return float(m.group(1))

    return np.nan


# Categorize display panel type
def display_panel(text):
    if pd.isna(text):
        return "Unknown"

    s = str(text).lower()

    if "amoled" in s:
        return "AMOLED"
    elif "oled" in s:
        return "OLED"
    elif "ips" in s:
        return "IPS LCD"
    elif "lcd" in s or "tft" in s:
        return "LCD/TFT"
    elif "monochrome" in s:
        return "Monochrome"
    else:
        return "Other"

# Group launch status
def launch_status_group(text):
    if pd.isna(text):
        return "Unknown"

    s = str(text).lower()

    if "available" in s:
        return "Available"
    elif "discontinued" in s:
        return "Discontinued"
    elif "cancelled" in s or "canceled" in s:
        return "Cancelled"
    elif "coming soon" in s:
        return "Coming soon"
    else:
        return "Other"

# Determine network generation
# Priority: 5G > 4G/LTE > 3G/HSPA > 2G/GSM
def network_generation(row):
    text_parts = []

    for col in ["network_technology", "network_speed", "network_5g_bands", "network_4g_bands", "network_3g_bands"]:
        if col in row.index and pd.notna(row[col]):
            text_parts.append(str(row[col]))

    text = " ".join(text_parts).lower()

    if "5g" in text or pd.notna(row.get("network_5g_bands", np.nan)):
        return 5
    elif "lte" in text or "4g" in text or pd.notna(row.get("network_4g_bands", np.nan)):
        return 4
    elif "hspa" in text or "umts" in text or "3g" in text or pd.notna(row.get("network_3g_bands", np.nan)):
        return 3
    elif "gsm" in text or "2g" in text:
        return 2
    else:
        return np.nan

# Keep major brands (frequency>=50) and group the rest as Other
def brand_group(oem):
    if pd.isna(oem):
        return "Unknown"

    top_brands = {
        "Samsung", "LG", "Motorola", "Nokia", "Huawei", "HTC", "Sony",
        "Sony Ericsson", "Asus", "Oppo", "Xiaomi", "vivo", "Honor",
        "Lenovo", "Apple", "Google", "OnePlus", "Realme", "ZTE", "alcatel"
    }

    if str(oem) in top_brands:
        return str(oem)

    return "Other"

# Extract price from misc_price and convert to EUR
# Prioritize EUR when multiple currencies exist
# Fixed conversion rates:
#    USD -> EUR: 0.93
#    GBP -> EUR: 1.17
#    INR -> EUR: 0.011
def parse_price_eur(text):
    if pd.isna(text):
        return np.nan

    s = clean_mojibake(text)
    s = s.replace(",", "")

    currency_patterns = [
        ("EUR", 1.0, [r"([\d.]+)\s*EUR\b", r"€\s*([\d.]+)"]),
        ("USD", 0.93, [r"\$\s*([\d.]+)", r"([\d.]+)\s*USD\b"]),
        ("GBP", 1.17, [r"£\s*([\d.]+)", r"([\d.]+)\s*GBP\b"]),
        ("INR", 0.011, [r"₹\s*([\d.]+)", r"([\d.]+)\s*INR\b"]),
    ]

    for currency, rate, patterns in currency_patterns:
        for pattern in patterns:
            m = re.search(pattern, s, flags=re.I)
            if m:
                return float(m.group(1)) * rate

    return np.nan

# Prevent division by zero or missing values
def safe_divide(a, b):
    return np.where((pd.notna(a)) & (pd.notna(b)) & (b != 0), a / b, np.nan)




# 5. Feature engineering

features = pd.DataFrame(index=df.index)

# Basic identifiers
features["oem"] = get_col("oem")
features["model"] = get_col("model")

# Brand-related features
features["brand_group"] = get_col("oem").map(brand_group)

premium_brands = {
    "Apple", "Samsung", "Google", "Sony", "Huawei",
    "OnePlus", "Xiaomi", "Oppo", "vivo", "Honor"
}

features["is_premium_brand"] = get_col("oem").map(
    lambda x: int(str(x) in premium_brands) if pd.notna(x) else 0
)

# Release information
features["launch_year"] = get_col("launch_announced").map(parse_year)
features["launch_month"] = get_col("launch_announced").map(parse_month)
features["phone_age"] = 2026 - features["launch_year"]
features["launch_status_group"] = get_col("launch_status").map(launch_status_group)

# Size and weight features
dims = get_col("body_dimensions").map(parse_dimensions_mm)
features[["body_height_mm", "body_width_mm", "body_thickness_mm"]] = pd.DataFrame(
    dims.tolist(), index=df.index
)

features["body_weight_g"] = get_col("body_weight").map(parse_weight_g)

# Display features
features["display_size_in"] = get_col("display_size").map(parse_display_size_in)
features["screen_to_body_pct"] = get_col("display_size").map(parse_screen_body_ratio)

res = get_col("display_resolution").map(parse_resolution)
features[["resolution_width_px", "resolution_height_px", "ppi_reported"]] = pd.DataFrame(
    res.tolist(), index=df.index
)

features["resolution_total_px"] = features["resolution_width_px"] * features["resolution_height_px"]

features["aspect_ratio"] = safe_divide(
    features[["resolution_width_px", "resolution_height_px"]].max(axis=1),
    features[["resolution_width_px", "resolution_height_px"]].min(axis=1)
)

features["ppi_calculated"] = safe_divide(
    np.sqrt(features["resolution_width_px"] ** 2 + features["resolution_height_px"] ** 2),
    features["display_size_in"]
)

features["ppi"] = features["ppi_reported"].fillna(features["ppi_calculated"])
features["display_panel"] = get_col("display_type").map(display_panel)
features["has_touchscreen"] = get_col("display_type").map(lambda x: contains_feature(x, ["touchscreen"]))

# Battery features
features["battery_capacity_mah"] = get_col("battery").map(parse_battery_mah)
features["battery_type"] = get_col("battery").map(parse_battery_type)
features["is_removable_battery"] = get_col("battery").map(parse_removable_battery)

features["fast_charging_w"] = get_col("battery_charging").map(parse_charging_w)

features["battery_per_inch"] = safe_divide(
    features["battery_capacity_mah"],
    features["display_size_in"]
)

features["battery_per_weight"] = safe_divide(
    features["battery_capacity_mah"],
    features["body_weight_g"]
)

# Memory features
memory_parsed = get_col("memory_internal").map(parse_memory)
features[["storage_gb", "ram_gb"]] = pd.DataFrame(memory_parsed.tolist(), index=df.index)

# Remove unrealistic storage values
features.loc[
    features["storage_gb"] < 0.5,
    "storage_gb"
] = np.nan

# Remove unrealistic RAM values
features.loc[
    features["ram_gb"] < 0.1,
    "ram_gb"
] = np.nan

features["has_card_slot"] = get_col("memory_card_slot").map(yes_no_feature)

# Camera features
main_camera_cols = [
    col for col in [
        "main_camera_single",
        "main_camera_dual",
        "main_camera_triple",
        "main_camera_quad",
        "main_camera_five",
        "main_camera_dual_or_triple",
        "main_camera"
    ]
    if col in df.columns
]

selfie_camera_cols = [
    col for col in [
        "selfie_camera_single",
        "selfie_camera_dual",
        "selfie_camera_triple",
        "selfie_camera"
    ]
    if col in df.columns
]

features["main_camera_count"] = 0

camera_count_map = {
    "main_camera_single": 1,
    "main_camera_dual": 2,
    "main_camera_dual_or_triple": 2,
    "main_camera_triple": 3,
    "main_camera_quad": 4,
    "main_camera_five": 5,
}

for col, count in camera_count_map.items():
    if col in df.columns:
        idx = df[col].notna()
        features.loc[idx, "main_camera_count"] = np.maximum(
            features.loc[idx, "main_camera_count"],
            count
        )

features["selfie_camera_count"] = 0

selfie_count_map = {
    "selfie_camera_single": 1,
    "selfie_camera_dual": 2,
    "selfie_camera_triple": 3,
}

for col, count in selfie_count_map.items():
    if col in df.columns:
        idx = df[col].notna()
        features.loc[idx, "selfie_camera_count"] = np.maximum(
            features.loc[idx, "selfie_camera_count"],
            count
        )

if len(main_camera_cols) > 0:
    features["main_camera_max_mp"] = df[main_camera_cols].apply(
        lambda row: max_mp_from_texts(*row.values), axis=1
    )

    main_camera_joined = df[main_camera_cols].apply(
        lambda row: " ".join([str(v) for v in row.values if pd.notna(v)]), axis=1
    )

    features["has_ois"] = main_camera_joined.map(lambda x: contains_feature(x, ["ois"]))
    features["has_ultrawide"] = main_camera_joined.map(lambda x: contains_feature(x, ["ultrawide"]))
    features["has_telephoto"] = main_camera_joined.map(
        lambda x: contains_feature(x, ["telephoto", "optical zoom", "periscope"])
    )
else:
    features["main_camera_max_mp"] = np.nan
    features["has_ois"] = 0
    features["has_ultrawide"] = 0
    features["has_telephoto"] = 0

if len(selfie_camera_cols) > 0:
    features["selfie_camera_max_mp"] = df[selfie_camera_cols].apply(
        lambda row: max_mp_from_texts(*row.values), axis=1
    )
else:
    features["selfie_camera_max_mp"] = np.nan

# Network and connectivity features
features["network_generation"] = df.apply(network_generation, axis=1)

features["has_5g"] = (features["network_generation"] >= 5).astype(int)
features["has_4g_or_more"] = (features["network_generation"] >= 4).astype(int)

features["has_wifi"] = get_col("comms_wlan").map(yes_no_feature)
features["has_bluetooth"] = get_col("comms_bluetooth").map(yes_no_feature)
features["has_gps"] = get_col("comms_gps").map(yes_no_feature)
features["has_nfc"] = get_col("comms_nfc").map(yes_no_feature)
features["has_radio"] = get_col("comms_radio").map(yes_no_feature)
features["has_usb"] = get_col("comms_usb").map(yes_no_feature)
features["has_3_5mm_jack"] = get_col("sound_3.5mm_jack").map(yes_no_feature)

# OS and sensor features
features["os_family"] = get_col("platform_os").map(os_family)
features["os_version"] = get_col("platform_os").map(parse_os_version)

features["sensor_count"] = get_col("features_sensors").map(count_sensors)
features["has_fingerprint"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["fingerprint"]))
features["has_face_id"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["face id"]))
features["has_accelerometer"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["accelerometer"]))
features["has_gyro"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["gyro"]))
features["has_proximity"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["proximity"]))
features["has_compass"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["compass"]))
features["has_barometer"] = get_col("features_sensors").map(lambda x: contains_feature(x, ["barometer"]))

# Target price
features["price_eur"] = get_col("misc_price").map(parse_price_eur)

# Price tier classification
features["price_tier"] = pd.cut(
    features["price_eur"],
    bins=[0, 150, 400, 800, np.inf],
    labels=["budget", "mid_range", "premium", "flagship"]
)




# 6. Outlier handling

valid_ranges = {
    "launch_year": (1980, 2026),
    "phone_age": (0, 60),
    "body_weight_g": (40, 500),
    "body_height_mm": (50, 250),
    "body_width_mm": (25, 150),
    "body_thickness_mm": (3, 40),
    "display_size_in": (1, 12),
    "screen_to_body_pct": (5, 100),
    "resolution_width_px": (50, 5000),
    "resolution_height_px": (50, 6000),
    "ppi": (20, 1000),
    "battery_capacity_mah": (300, 20000),
    "fast_charging_w": (0, 250),
    "storage_gb": (0, 2048),
    "ram_gb": (0, 32),
    "main_camera_count": (0, 5),
    "selfie_camera_count": (0, 3),
    "main_camera_max_mp": (0, 300),
    "selfie_camera_max_mp": (0, 200),
    "network_generation": (2, 5),
    "sensor_count": (0, 20),
    "price_eur": (10, 5000),
}

for col, (low, high) in valid_ranges.items():
    if col in features.columns:
        features.loc[(features[col] < low) | (features[col] > high), col] = np.nan




# 7. Value-for-money recommendation features

def minmax_scale_series(s):
    """
    Min-max scaling for recommendation score calculation
    """
    s = pd.to_numeric(s, errors="coerce")

    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=s.index)

    if s.max() == s.min():
        return pd.Series(0.5, index=s.index)

    return (s - s.min()) / (s.max() - s.min())


score_weights = {
    "battery_capacity_mah": 0.14,
    "ram_gb": 0.16,
    "storage_gb": 0.11,
    "ppi": 0.10,
    "main_camera_max_mp": 0.09,
    "selfie_camera_max_mp": 0.05,
    "network_generation": 0.10,
    "sensor_count": 0.05,
    "screen_to_body_pct": 0.04,
    "fast_charging_w": 0.05,
    "os_version": 0.05,

    # add recency factor
    "launch_year": 0.15,
}

score_scaled = pd.DataFrame(index=features.index)

for col in score_weights.keys():
    if col in features.columns:
        score_scaled[col] = minmax_scale_series(features[col])

score_scaled = score_scaled.fillna(score_scaled.median(numeric_only=True))

weights = pd.Series(score_weights)
weights = weights[score_scaled.columns]

features["spec_score_0_100"] = (
    score_scaled.mul(weights, axis=1).sum(axis=1) / weights.sum() * 100
)

features["value_score"] = safe_divide(features["spec_score_0_100"], features["price_eur"]) * 100
features["price_per_ram_gb"] = safe_divide(features["price_eur"], features["ram_gb"])
features["price_per_storage_gb"] = safe_divide(features["price_eur"], features["storage_gb"])
features["battery_per_eur"] = safe_divide(features["battery_capacity_mah"], features["price_eur"])

features = features.replace([np.inf, -np.inf], np.nan)

print("\n[INFO] Engineered feature shape:", features.shape)

try:
    display(features.head())
except:
    print(features.head())




# 8. Build dataset for price prediction

# Columns excluded from price prediction:
# - oem, model: 고유값이 많고 모델 일반화에 방해 가능
# - price_tier: price_eur에서 파생된 컬럼
# - spec_score/value_score/price_per_*: 가격 또는 가격 기반 추천용 파생 피처
leakage_or_id_cols = [
    "oem",
    "model",
    "price_tier",
    "spec_score_0_100",
    "value_score",
    "price_per_ram_gb",
    "price_per_storage_gb",
    "battery_per_eur"
]

price_prediction_df = features.drop(columns=leakage_or_id_cols, errors="ignore").copy()

# Remove rows without the target price (price_eur)
price_prediction_df = price_prediction_df[
    price_prediction_df["price_eur"].notna()
].reset_index(drop=True)

print("\n[INFO] Price prediction feature table shape:", price_prediction_df.shape)

try:
    display(price_prediction_df.head())
except:
    print(price_prediction_df.head())





# 9. Build dataset for smartphone recommendation

recommendation_cols = [
    "oem",
    "model",
    "brand_group",
    "price_eur",
    "price_tier",

    "launch_year",
    "phone_age",
    "launch_status_group",

    "body_weight_g",
    "display_size_in",
    "screen_to_body_pct",
    "ppi",
    "aspect_ratio",

    "battery_capacity_mah",
    "fast_charging_w",
    "battery_per_inch",
    "battery_per_weight",

    "ram_gb",
    "storage_gb",
    "has_card_slot",

    "main_camera_count",
    "main_camera_max_mp",
    "selfie_camera_count",
    "selfie_camera_max_mp",
    "has_ois",
    "has_ultrawide",
    "has_telephoto",

    "network_generation",
    "has_5g",
    "has_4g_or_more",

    "display_panel",
    "battery_type",
    "os_family",

    "has_wifi",
    "has_bluetooth",
    "has_gps",
    "has_nfc",
    "has_radio",
    "has_usb",
    "has_3_5mm_jack",

    "sensor_count",
    "has_fingerprint",
    "has_face_id",
    "has_accelerometer",
    "has_gyro",
    "has_proximity",
    "has_compass",
    "has_barometer",

    "spec_score_0_100",
    "value_score",
    "price_per_ram_gb",
    "price_per_storage_gb",
    "battery_per_eur"
]

recommendation_cols = [c for c in recommendation_cols if c in features.columns]

recommendation_df = features[recommendation_cols].copy()

# Remove rows without price information
# Filter out very cheap legacy feature phones
recommendation_df = recommendation_df[
    recommendation_df["price_eur"] >= 50
].reset_index(drop=True)

# Remove devices with extremely low RAM
recommendation_df = recommendation_df[
    recommendation_df["ram_gb"] >= 2
].reset_index(drop=True)

recommendation_df = recommendation_df.sort_values(
    "value_score",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

print("\n[INFO] Recommendation feature table shape:", recommendation_df.shape)

try:
    display(recommendation_df.head(20))
except:
    print(recommendation_df.head(20))



# 10. Encoding and scaling function

def make_encoded_scaled_csv(
    input_df,
    output_path,
    target_col=None,
    id_cols=None,
    drop_cols=None
):
    """
Numeric features:
- Replace missing values with the median
- Apply StandardScaler

Categorical features:
- Replace missing values with the most frequent value
- Apply OneHotEncoder

target_col:
- If a target variable exists,preserve it as the first column without scaling

id_cols:
- Exclude identifier columns such as oem and model from model inputs

drop_cols:
- Additional columns to exclude
"""

    id_cols = id_cols or []
    drop_cols = drop_cols or []

    data = input_df.copy()

    y = None
    if target_col is not None and target_col in data.columns:
        y = data[target_col].copy()
        data = data.drop(columns=[target_col])

    remove_cols = [c for c in id_cols + drop_cols if c in data.columns]
    data = data.drop(columns=remove_cols, errors="ignore")

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = data.select_dtypes(exclude=[np.number]).columns.tolist()

    print("\n[ENCODING/SCALING]", output_path)
    print("Numeric columns:", len(numeric_cols))
    print("Categorical columns:", len(categorical_cols))

    # Handle OneHotEncoder version compatibility
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", onehot)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols)
        ],
        remainder="drop"
    )

    X_processed = preprocessor.fit_transform(data)

    try:
        processed_feature_names = preprocessor.get_feature_names_out()
    except:
        processed_feature_names = [f"feature_{i}" for i in range(X_processed.shape[1])]

    processed_df = pd.DataFrame(
        X_processed,
        columns=processed_feature_names
    )

    if y is not None:
        processed_df.insert(0, f"target_{target_col}", y.values)

    processed_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("[SAVED]", output_path)
    print("[INFO] Encoded/scaled shape:", processed_df.shape)

    return processed_df



# 11. Save final CSV files

# 11-1. Feature-engineered dataset for price prediction
price_prediction_df.to_csv(
    PRICE_PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n[SAVED]", PRICE_PROCESSED_PATH)

# 11-2. Feature-engineered dataset for smartphone recommendation
recommendation_df.to_csv(
    RECO_PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("[SAVED]", RECO_PROCESSED_PATH)

# 11-3. Encoded and scaled dataset for price prediction
price_all_df = make_encoded_scaled_csv(
    input_df=price_prediction_df,
    output_path=PRICE_ALL_PATH,
    target_col="price_eur",
    id_cols=[],
    drop_cols=[]
)

# 11-4. Encoded and scaled dataset for recommendation
# Exclude oem and model because they are identifier strings
# brand_group, price_tier, display_panel, battery_type, os_family..(categorical features) -> One-Hot Encoding
recommendation_all_df = make_encoded_scaled_csv(
    input_df=recommendation_df,
    output_path=RECO_ALL_PATH,
    target_col=None,
    id_cols=["oem", "model"],
    drop_cols=[]
)



print("\n========== 최종 ==========")
print("1.", PRICE_PROCESSED_PATH)
print("2.", RECO_PROCESSED_PATH)
print("3.", PRICE_ALL_PATH)
print("4.", RECO_ALL_PATH)

print("\n========== 요약 ==========")
print("원본 데이터 크기:", df_raw.shape)
print("중복 제거 후 크기:", df.shape)
print("전체 피처 테이블 크기:", features.shape)
print("가격 예측용 전처리+피처엔지니어링 데이터 크기:", price_prediction_df.shape)
print("추천용 전처리+피처엔지니어링 데이터 크기:", recommendation_df.shape)
print("가격 예측용 인코딩+스케일링 데이터 크기:", price_all_df.shape)
print("추천/군집화용 인코딩+스케일링 데이터 크기:", recommendation_all_df.shape)

print("\n========== 저장 위치 ==========")
print("/content/gsm_processed(price_prediction).csv")
print("/content/gsm_processed(recommendation).csv")
print("/content/gsm_processed_all(price_prediction).csv")
print("/content/gsm_processed_all(recommendation).csv")

[INFO] Loaded file: /content/gsm/gsm.csv
[INFO] Raw shape: (10679, 86)


,oem,model,network_technology,network_2g_bands,network_gprs,network_edge,launch_announced,launch_status,body_dimensions,body_weight,...,main_camera_dual_or_triple,battery_music_play,selfie_camera_triple,main_camera_v1,selfie_camera,camera,main_camera,network,battery_talk_time,battery_stand.by
0,Benefon,Vega,GSM,GSM 900,No,No,1999,Discontinued,145 x 56 x 23 mm (5.71 x 2.20 x 0.91 in),190 g (6.70 oz),...,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN,4 - 10 h,3 - 6 days
1,Garmin-Asus,nuvifone M10,GSM / HSPA,GSM 900 / 1800 / 1900,NaN,NaN,"2010, January. Released 2010, March",Discontinued,-,-,...,NaN,NaN,NaN,NaN,V2,NaN,NaN,GSM 850 / 1800 / 1900 - US version,Up to 8 h,Up to 600 h (2G) / Up to 600 h (3G)
2,Gigabyte,GSmart G1305 Boston,GSM / HSPA,GSM 850 / 900 / 1800 / 1900,NaN,NaN,"2010, April. Released 2010, April",Discontinued,116 x 56.8 x 12.4 mm (4.57 x 2.24 x 0.49 in),118 g (4.16 oz),...,NaN,NaN,NaN,NaN,V2,NaN,NaN,NaN,Up to 7 h 10 min,Up to 410 h
3,Gigabyte,GSmart,GSM / HSPA,GSM 900 / 1800,NaN,NaN,Not officially announced yet,Cancelled,103 x 54 x 13.4 mm (4.06 x 2.13 x 0.53 in),-,...,NaN,NaN,NaN,NaN,V2,NaN,NaN,NaN,NaN,NaN
4,Google,Pixel 4 XL,GSM / CDMA / HSPA / EVDO / LTE,GSM 850 / 900 / 1800 / 1900,NaN,NaN,"2019, October 15","Available. Released 2019, October 22",160.4 x 75.1 x 8.2 mm (6.31 x 2.96 x 0.32 in),193 g (6.81 oz),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CDMA 800 / 1900,NaN,NaN



[INFO] Columns
['oem', 'model', 'network_technology', 'network_2g_bands', 'network_gprs', 'network_edge', 'launch_announced', 'launch_status', 'body_dimensions', 'body_weight', 'body_sim', 'display_type', 'display_size', 'display_resolution', 'display', 'memory_card_slot', 'memory_phonebook', 'memory_call_records', 'sound_loudspeaker', 'sound_alert_types', 'sound_3.5mm_jack', 'comms_wlan', 'comms_bluetooth', 'comms_gps', 'comms_radio', 'comms_usb', 'features_sensors', 'features_messaging', 'features_browser', 'features_clock', 'features_alarm', 'features_games', 'features_java', 'features', 'misc_colors', 'network_3g_bands', 'network_speed', 'platform_os', 'platform_chipset', 'platform_cpu', 'platform_gpu', 'memory_internal', 'main_camera_single', 'main_camera_video', 'misc_price', 'main_camera_features', 'body', 'network_4g_bands', 'body_build', 'display_protection', 'memory', 'main_camera_dual', 'selfie_camera_dual', 'selfie_camera_features', 'selfie_camera_video', 'comms_nfc', 'bat

,0
oem,object
model,object
network_technology,object
network_2g_bands,object
network_gprs,object
...,...
camera,object
main_camera,object
network,object
battery_talk_time,object



[INFO] Missing values - top 20


...


[INFO] Price prediction feature table shape: (6251, 58)


,brand_group,is_premium_brand,launch_year,launch_month,phone_age,launch_status_group,body_height_mm,body_width_mm,body_thickness_mm,body_weight_g,...,os_version,sensor_count,has_fingerprint,has_face_id,has_accelerometer,has_gyro,has_proximity,has_compass,has_barometer,price_eur
0,Other,0,2010.0,1.0,16.0,Discontinued,NaN,NaN,NaN,NaN,...,6.5,1.0,0,0,1,0,0,0,0,310.00
1,Other,0,2010.0,4.0,16.0,Discontinued,116.0,56.8,12.4,118.0,...,1.6,1.0,0,0,1,0,0,0,0,110.00
2,Google,1,2019.0,10.0,7.0,Available,160.4,75.1,8.2,193.0,...,10.0,6.0,0,1,1,1,1,1,1,599.00
3,Google,1,2019.0,10.0,7.0,Available,147.1,68.8,8.2,162.0,...,10.0,6.0,0,1,1,1,1,1,1,524.52
4,Google,1,2019.0,5.0,7.0,Available,160.1,76.1,8.2,167.0,...,9.0,6.0,1,0,1,1,1,1,1,329.00



[INFO] Recommendation feature table shape: (2113, 53)


,oem,model,brand_group,price_eur,price_tier,launch_year,phone_age,launch_status_group,body_weight_g,display_size_in,...,has_accelerometer,has_gyro,has_proximity,has_compass,has_barometer,spec_score_0_100,value_score,price_per_ram_gb,price_per_storage_gb,battery_per_eur
0,Motorola,Moto E6,Motorola,55.7907,budget,2019.0,7.0,Available,159.0,5.50,...,1,0,1,1,0,33.047602,59.234966,27.895350,3.486919,53.772403
1,LG,Aristo 2,LG,55.8000,budget,2018.0,8.0,Available,139.0,5.00,...,1,0,1,0,0,30.960627,55.484994,27.900000,3.487500,43.189964
2,Micromax,Bharat 4 Q440,Other,55.0000,budget,2017.0,9.0,Available,150.0,5.00,...,1,0,0,0,0,30.311743,55.112260,18.333333,3.437500,45.454545
3,Nokia,C2 Tennen,Nokia,60.0000,budget,2020.0,6.0,Available,180.0,5.45,...,1,0,1,0,0,31.832843,53.054738,30.000000,1.875000,50.000000
4,Panasonic,P100,Other,58.3000,budget,2018.0,8.0,Available,174.8,5.00,...,1,0,1,0,0,30.837786,52.895002,29.150000,3.643750,37.735849
5,Coolpad,Cool 3 Plus,Other,66.0000,budget,2019.0,7.0,Available,135.0,5.71,...,1,0,1,0,0,34.311479,51.987089,22.000000,2.062500,45.454545
6,Lava,Z61,Other,62.7000,budget,2018.0,8.0,Available,139.0,5.45,...,1,0,1,0,0,31.363179,50.021020,31.350000,3.918750,47.846890
7,YU,Ace,Other,66.0000,budget,2018.0,8.0,Available,NaN,5.45,...,1,0,1,0,0,32.473634,49.202475,33.000000,2.062500,60.606061
8,Micromax,Canvas Xpress 4G Q413,Other,60.0000,budget,2015.0,11.0,Available,155.0,5.00,...,1,0,1,0,0,28.379701,47.299502,30.000000,3.750000,33.333333
9,Panasonic,Eluga I7,Other,71.5000,budget,2018.0,8.0,Available,168.0,5.45,...,1,0,1,1,0,33.245542,46.497261,35.750000,4.468750,55.944056



[SAVED] /content/gsm_processed(price_prediction).csv
[SAVED] /content/gsm_processed(recommendation).csv

[ENCODING/SCALING] /content/gsm_processed_all(price_prediction).csv
Numeric columns: 52
Categorical columns: 5
[SAVED] /content/gsm_processed_all(price_prediction).csv
[INFO] Encoded/scaled shape: (6251, 95)

[ENCODING/SCALING] /content/gsm_processed_all(recommendation).csv
Numeric columns: 45
Categorical columns: 6
[SAVED] /content/gsm_processed_all(recommendation).csv
[INFO] Encoded/scaled shape: (2113, 86)

========== 최종 ==========
1. /content/gsm_processed(price_prediction).csv
2. /content/gsm_processed(recommendation).csv
3. /content/gsm_processed_all(price_prediction).csv
4. /content/gsm_processed_all(recommendation).csv

========== 요약 ==========
원본 데이터 크기: (10679, 86)
중복 제거 후 크기: (10105, 86)
전체 피처 테이블 크기: (10105, 66)
가격 예측용 전처리+피처엔지니어링 데이터 크기: (6251, 58)
추천용 전처리+피처엔지니어링 데이터 크기: (2113, 53)
가격 예측용 인코딩+스케일링 데이터 크기: (6251, 95)
추천/군집화용 인코딩+스케일링 데이터 크기: (2113, 86)

========== 저장 위

## Check dataset structure

In [ ]:
# GSM smartphone dataset column overview


import pandas as pd
from IPython.display import display


df = pd.read_csv('/content/gsm.csv')

print("="*60)
print("데이터셋 기본 정보")
print("="*60)

print(f"행(Row) 개수 : {df.shape[0]}")
print(f"열(Column) 개수 : {df.shape[1]}")

print("\n컬럼 목록:")

for col in df.columns:
    print(col)


# Categorize columns by domain

column_groups = {

    "1. 제조사 및 모델 정보": [
        "oem",
        "model"
    ],

    "2. 네트워크 관련 정보": [
        col for col in df.columns
        if col.startswith("network")
    ],

    "3. 출시 정보": [
        col for col in df.columns
        if col.startswith("launch")
    ],

    "4. 디자인 / 바디 정보": [
        col for col in df.columns
        if col.startswith("body")
    ],

    "5. 디스플레이 정보": [
        col for col in df.columns
        if col.startswith("display")
    ],

    "6. 플랫폼 및 성능 정보": [
        col for col in df.columns
        if col.startswith("platform")
    ],

    "7. 메모리 정보": [
        col for col in df.columns
        if col.startswith("memory")
    ],

    "8. 카메라 정보": [
        col for col in df.columns
        if "camera" in col
    ],

    "9. 배터리 정보": [
        col for col in df.columns
        if "battery" in col
    ],

    "10. 통신 및 기타 기능": [
        col for col in df.columns
        if col.startswith("comms")
    ],

    "11. 가격 및 기타 정보": [
        col for col in df.columns
        if col.startswith("misc")
    ]
}


# Display column groups

print("\n")
print("="*60)
print("컬럼 카테고리 분류")
print("="*60)

for category, cols in column_groups.items():

    print(f"\n[{category}]")
    print("-"*50)

    if len(cols) == 0:
        print("해당 컬럼 없음")
        continue

    for col in cols:
        print(f"- {col}")

# Display detailed column information

print("\n")
print("="*60)
print("컬럼 상세 정보")
print("="*60)

column_info = []

for col in df.columns:

    dtype = df[col].dtype

    missing = df[col].isnull().sum()

    missing_ratio = round(
        (missing / len(df)) * 100,
        2
    )

    # Extract up to three sample values
    sample_values = (
        df[col]
        .dropna()
        .astype(str)
        .unique()[:3]
    )

    sample_values = ", ".join(sample_values)

    column_info.append({

        "Column": col,
        "DataType": str(dtype),
        "MissingCount": missing,
        "MissingRatio(%)": missing_ratio,
        "SampleValues": sample_values
    })

info_df = pd.DataFrame(column_info)

display(info_df)


# Display the number of columns in each category

print("\n")
print("="*60)
print("카테고리별 컬럼 개수")
print("="*60)

for category, cols in column_groups.items():

    print(f"{category} : {len(cols)}개")


# Top 20 columns with the most missing values

print("\n")
print("="*60)
print("결측치 많은 컬럼 TOP 20")
print("="*60)

missing_df = pd.DataFrame({

    "Column": df.columns,

    "MissingCount": df.isnull().sum().values,

    "MissingRatio(%)": (
        df.isnull().sum().values / len(df) * 100
    ).round(2)

})

missing_df = missing_df.sort_values(
    by="MissingCount",
    ascending=False
)

display(missing_df.head(20))


데이터셋 기본 정보
행(Row) 개수 : 10679
열(Column) 개수 : 86

컬럼 목록:
oem
model
network_technology
network_2g_bands
network_gprs
network_edge
launch_announced
launch_status
body_dimensions
body_weight
body_sim
display_type
display_size
display_resolution
display
memory_card_slot
memory_phonebook
memory_call_records
sound_loudspeaker
sound_alert_types
sound_3.5mm_jack
comms_wlan
comms_bluetooth
comms_gps
comms_radio
comms_usb
features_sensors
features_messaging
features_browser
features_clock
features_alarm
features_games
features_java
features
misc_colors
network_3g_bands
network_speed
platform_os
platform_chipset
platform_cpu
platform_gpu
memory_internal
main_camera_single
main_camera_video
misc_price
main_camera_features
body
network_4g_bands
body_build
display_protection
memory
main_camera_dual
selfie_camera_dual
selfie_camera_features
selfie_camera_video
comms_nfc
battery_charging
misc_models
tests_performance
tests_camera
tests_loudspeaker
tests_audio_quality
tests_battery_life
tests_display
sel

,Column,DataType,MissingCount,MissingRatio(%),SampleValues
0,oem,object,0,0.00,"Benefon, Garmin-Asus, Gigabyte"
1,model,object,0,0.00,"Vega, nuvifone M10, GSmart G1305 Boston"
2,network_technology,object,0,0.00,"GSM, GSM / HSPA, GSM / CDMA / HSPA / EVDO / LTE"
3,network_2g_bands,object,324,3.03,"GSM 900, GSM 900 / 1800 / 1900, GSM 850 / 900 ..."
4,network_gprs,object,965,9.04,"No, Class 10, Class 8"
...,...,...,...,...,...
81,camera,object,9333,87.40,"No, V2, VGA"
82,main_camera,object,10678,99.99,No
83,network,object,8159,76.40,"GSM 850 / 1800 / 1900 - US version, CDMA 800 /..."
84,battery_talk_time,object,3401,31.85,"4 - 10 h, Up to 8 h, Up to 7 h 10 min"




카테고리별 컬럼 개수
1. 제조사 및 모델 정보 : 2개
2. 네트워크 관련 정보 : 9개
3. 출시 정보 : 2개
4. 디자인 / 바디 정보 : 6개
5. 디스플레이 정보 : 5개
6. 플랫폼 및 성능 정보 : 4개
7. 메모리 정보 : 5개
8. 카메라 정보 : 18개
9. 배터리 정보 : 6개
10. 통신 및 기타 기능 : 7개
11. 가격 및 기타 정보 : 5개


결측치 많은 컬럼 TOP 20


,Column,MissingCount,MissingRatio(%)
82,main_camera,10678,99.99
76,main_camera_dual_or_triple,10676,99.97
78,selfie_camera_triple,10675,99.96
71,main_camera_five,10673,99.94
79,main_camera_v1,10672,99.93
52,selfie_camera_dual,10528,98.59
66,network_5g_bands,10513,98.45
67,main_camera_quad,10445,97.81
72,features_languages,10376,97.16
68,main_camera_triple,10368,97.09
